# Station Training Baseline — RKSI: Seoul — Incheon International Airport

**Status: active baseline.** This is the complete per-station workflow: the **Seoul V20 Asia no-peak aligned** point pipeline, followed in this same notebook by **Seoul 1C Market Ordinal Probability Model**, the pure cumulative-threshold ordinal probability model. Versioned source notebooks remain reference-only; new station work starts here.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "asia_station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/asia_station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CITY_ID = "seoul"
CITY_LABEL = "Seoul"
STATION_ID = "RKSI"
TIMEZONE = "Asia/Seoul"
DATA_ROOT = PROJECT_ROOT / "data" / "calibration" / "asia_11am"
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_training_baseline" / "Seoul"
PROVIDERS = ("gfs", "gefs", "jma_msm")
TIMING_MODE = "asia_same_day_11am_live_safe"
FEATURE_VERSION = "v20_asia_no_peak"
TRAINING_PROFILE = "v20_aligned"
TARGET_SOURCE = "wunderground_only"
TARGET_MODE = "remaining_warmup"
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODEL_WEIGHTS = True
PROBABILITY_MODEL_VERSION = "station_bucket_baseline_seoul_1c_market_ordinal"
PROBABILITY_TARGET = "celsius_market_1c"
PROBABILITY_OUTPUT_SUBDIR = "celsius_market_probability"
PROBABILITY_FEATURE_PROFILE = "asia_no_peak"
PROBABILITY_FEATURE_COUNT = 59
PROBABILITY_PROVIDERS = ('gfs', 'gefs', 'jma_msm')
PROBABILITY_DEVELOPMENT_YEARS = (2024, 2025)
PROBABILITY_FORWARD_VALIDATION_YEARS = (2025,)
PROBABILITY_HOLDOUT_YEAR = 2026
MODEL_VERSION = f"station_high_regressor_baseline_seoul_no_peak_stack"
PROJECT_ROOT


In [ ]:
import numpy as np
import pandas as pd

from src.calibration.asia_station_stacking import (
    ASIA_PROVIDERS,
    ASIA_TEST_YEAR,
    ASIA_TIMING_MODE,
    asia_expanding_folds,
    build_asia_station_wide_dataset,
    provider_readiness,
)
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_ASIA_NO_PEAK_FEATURE_VERSION,
    missing_model_dependencies,
    run_station_year_split_experiment,
)
from src.export_station_stacking_v2_models import export_station_model_weights


## City contract

- Existing Asia parquet data rooted at `data/calibration/asia_11am`
- Local 11 AM live-safe observation cutoff
- GFS, GEFS, and JMA MSM forecast inputs
- Wunderground-only daily settlement high target
- Fahrenheit-native model values with Celsius reporting


In [ ]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
            "validation_weight": 1.0,
        }
        for fold in asia_expanding_folds()
    ]
)
fold_spec


## Provider readiness


In [ ]:
readiness = provider_readiness(DATA_ROOT, CITY_ID, providers=PROVIDERS)
readiness


## Build the live-safe modeling frame


In [ ]:
features = build_asia_station_wide_dataset(
    DATA_ROOT,
    CITY_ID,
    feature_version=FEATURE_VERSION,
    providers=PROVIDERS,
)
features[[
    "contract_date",
    "actual_high_f",
    "observed_high_temp_through_as_of_f",
    "gfs_high_f",
    "gefs_high_f",
    "jma_msm_high_f",
    "strict_quality_ok",
]].head()


In [ ]:
feature_coverage = (
    features[["actual_high_f", "observed_high_temp_through_as_of_f", *[f"{p}_high_f" for p in PROVIDERS]]]
    .notna()
    .mean()
    .rename("non_null_fraction")
    .to_frame()
)
feature_coverage


## Train and score


In [ ]:
missing_packages = missing_model_dependencies(("xgboost", "lightgbm", "catboost", "optuna"))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric="mae_f",
    feature_version=FEATURE_VERSION,
    training_profile=TRAINING_PROFILE,
    target_mode=TARGET_MODE,
    target_source=TARGET_SOURCE,
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=asia_expanding_folds(),
    year_split_validation_weights={2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2022, 2025),
    year_split_test_year=ASIA_TEST_YEAR,
    output_dir=OUTPUT_DIR,
    prebuilt_features=features,
)
config.resolved_optuna_storage_path()


In [ ]:
result = run_station_year_split_experiment(config)
result.scoreboard


## Celsius reporting and export


In [ ]:
celsius_predictions = result.test_predictions.copy()
for column in ("actual_high_f", "predicted_high_f", "error_f"):
    if column in celsius_predictions:
        celsius_predictions[column.replace("_f", "_c")] = pd.to_numeric(celsius_predictions[column], errors="coerce") * 5.0 / 9.0
celsius_predictions.head()


In [ ]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_training_baseline/stations/Seoul",
    )
    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this notebook.")


In [ ]:
result.output_paths


## Seoul 1C Market Ordinal Probability Model — market-aligned correction

This stage replaces Seoul's historical integer-Fahrenheit/2°F probability
target with the actual Seoul Polymarket whole-1°C market contract. The point
model remains Fahrenheit-native.

- `point_prediction_c = (point_prediction_f - 32) * 5 / 9`;
- `point_bucket_c = floor(point_prediction_c + 0.5)`;
- `actual_bucket_c = floor(actual_high_c + 0.5)`;
- `offset_c = actual_bucket_c - point_bucket_c`;
- ordered classes: `<=-3, -2, -1, 0, +1, +2, >=+3` °C, chosen from pre-2026
  support with open tails;
- exact market probabilities are `point_bucket_c + exact_offset_c` and sum to
  one on every row;
- all confidence thresholds and the tail-ambiguity rule are selected on the
  [2025] forward-validation rows only;
- 2026 remains exploratory and cannot select the model or policy.

The source frame has no settlement-equivalent `actual_high_c` or
`settlement_high_c` field. Its target is Wunderground `actual_high_f`, while
`iem_daily_high_c` is diagnostic and a different source. Therefore the Celsius
target uses the exact Fahrenheit-to-Celsius conversion fallback. This matches
Seoul Polymarket's integer Celsius settlement buckets without approximating the
old 2°F distribution.


In [ ]:
import json

from src.calibration.celsius_market_probability import (
    OFFSET_LABELS_C,
    TARGET_CONTRACT,
    build_celsius_probability_frame,
    celsius_calibration_table,
    celsius_probability_metrics,
    evaluate_celsius_probability_holdout,
    export_celsius_probability_bundle,
    fit_celsius_probability_system,
    sha256_file as celsius_sha256_file,
)
from src.calibration.bucket_probability import probability_feature_names
from src.calibration.v19_bucket import crossfit_ridge_predictions


### Celsius feature and target contract


In [ ]:
celsius_feature_names = probability_feature_names(
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
celsius_feature_contract = pd.DataFrame(
    {"position": range(1, len(celsius_feature_names) + 1), "feature": celsius_feature_names}
)
celsius_target_contract = {
    "market": "Seoul Polymarket whole 1C integer buckets",
    "rounding": "round_half_up(value) = floor(value + 0.5)",
    "target": TARGET_CONTRACT,
    "actual_celsius_source_used": "actual_high_f_converted_to_c",
    "excluded_diagnostic_source": "iem_daily_high_c",
    "ordered_offset_classes_c": list(OFFSET_LABELS_C),
    "tail_contract": "training-supported exact offsets within <=-3 and >=+3",
    "feature_profile": PROBABILITY_FEATURE_PROFILE,
    "feature_count": len(celsius_feature_names),
    "providers": list(PROBABILITY_PROVIDERS),
}
assert len(celsius_feature_names) == PROBABILITY_FEATURE_COUNT
celsius_target_contract


### Fit with chronological [2025] outer validation


In [ ]:
point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

celsius_training_frame = build_celsius_probability_frame(
    result.features,
    point_forward_predictions,
    result.validation_predictions,
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
assert celsius_training_frame["actual_high_c_source"].eq(
    "actual_high_f_converted_to_c"
).all()
celsius_bundle, celsius_forward_predictions, celsius_tuning = (
    fit_celsius_probability_system(
        celsius_training_frame,
        station_id=STATION_ID,
        point_model_version=MODEL_VERSION,
        point_bundle_sha256=celsius_sha256_file(exported_weights.bundle_path),
        feature_profile=PROBABILITY_FEATURE_PROFILE,
        model_version=PROBABILITY_MODEL_VERSION,
        development_years=PROBABILITY_DEVELOPMENT_YEARS,
        forward_validation_years=PROBABILITY_FORWARD_VALIDATION_YEARS,
    )
)
assert celsius_bundle["selected_family"] == "celsius_offset_ordinal_logistic"
assert celsius_bundle["training_cutoff"] < f"{PROBABILITY_HOLDOUT_YEAR}-01-01"
celsius_probability_metrics(celsius_forward_predictions)


### Verify chronology and exact Celsius market probabilities


In [ ]:
celsius_forward_dates = pd.to_datetime(celsius_forward_predictions["contract_date"])
assert set(celsius_forward_predictions["validation_year"]) == set(
    PROBABILITY_FORWARD_VALIDATION_YEARS
)
assert (
    pd.to_datetime(celsius_forward_predictions["model_training_cutoff"])
    < celsius_forward_dates
).all()
assert (
    pd.to_datetime(celsius_forward_predictions["calibration_training_cutoff"])
    < pd.to_datetime(celsius_forward_predictions["calibration_validation_start"])
).all()
assert (
    pd.to_datetime(celsius_forward_predictions["calibration_validation_cutoff"])
    < celsius_forward_dates
).all()
for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
    assert celsius_forward_predictions[column].map(
        lambda probabilities: np.isclose(sum(probabilities.values()), 1.0, atol=1e-10)
    ).all()
assert celsius_forward_predictions.apply(
    lambda row: int(row["recommended_bucket_c"])
    == int(max(row["market_bucket_probabilities_c"], key=row["market_bucket_probabilities_c"].get)),
    axis=1,
).all()
assert celsius_bundle["policy_selection_data"] == "pre-2026 forward validation only"
celsius_bundle["decision_thresholds"]


### Evaluate the frozen model on exploratory 2026


In [ ]:
holdout_point_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not holdout_point_predictions.empty

celsius_holdout_predictions, celsius_holdout_metrics, celsius_holdout_calibration = (
    evaluate_celsius_probability_holdout(
        result.features,
        holdout_point_predictions,
        result.test_predictions,
        celsius_bundle,
        holdout_year=PROBABILITY_HOLDOUT_YEAR,
    )
)
assert not celsius_holdout_predictions.empty
assert pd.to_datetime(celsius_holdout_predictions["contract_date"]).dt.year.eq(
    PROBABILITY_HOLDOUT_YEAR
).all()
for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
    assert celsius_holdout_predictions[column].map(
        lambda probabilities: np.isclose(sum(probabilities.values()), 1.0, atol=1e-10)
    ).all()
celsius_bundle["holdout_metrics"] = celsius_holdout_metrics.iloc[0].to_dict()
celsius_bundle["holdout_status"] = "exploratory_previously_inspected_shadow_only"
celsius_holdout_metrics


### Export the isolated Celsius research artifacts


In [ ]:
celsius_output_dir = config.resolved_output_dir() / PROBABILITY_OUTPUT_SUBDIR
celsius_output_dir.mkdir(parents=True, exist_ok=True)

def _serialize_probability_columns(frame):
    output = frame.copy()
    for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
        output[column] = output[column].map(lambda value: json.dumps(value, sort_keys=True))
    return output

celsius_artifact_paths = []
forward_predictions_path = celsius_output_dir / f"{STATION_ID}_forward_validation_predictions.csv"
forward_metrics_path = celsius_output_dir / f"{STATION_ID}_forward_validation_metrics.csv"
holdout_predictions_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_predictions.csv"
holdout_metrics_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_metrics.csv"
forward_calibration_path = celsius_output_dir / f"{STATION_ID}_forward_validation_calibration.csv"
holdout_calibration_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_calibration.csv"
tuning_path = celsius_output_dir / f"{STATION_ID}_pre_2026_tuning.csv"
feature_contract_path = celsius_output_dir / f"{STATION_ID}_celsius_feature_contract.csv"
target_contract_path = celsius_output_dir / f"{STATION_ID}_celsius_target_contract.json"

_serialize_probability_columns(celsius_forward_predictions).to_csv(forward_predictions_path, index=False)
celsius_probability_metrics(celsius_forward_predictions).to_csv(forward_metrics_path, index=False)
_serialize_probability_columns(celsius_holdout_predictions).to_csv(holdout_predictions_path, index=False)
celsius_holdout_metrics.to_csv(holdout_metrics_path, index=False)
celsius_calibration_table(celsius_forward_predictions).to_csv(forward_calibration_path, index=False)
celsius_holdout_calibration.to_csv(holdout_calibration_path, index=False)
celsius_tuning.to_csv(tuning_path, index=False)
celsius_feature_contract.to_csv(feature_contract_path, index=False)
target_contract_path.write_text(json.dumps(celsius_target_contract, indent=2, sort_keys=True) + "\n", encoding="utf-8")
celsius_artifact_paths.extend([
    forward_predictions_path, forward_metrics_path, holdout_predictions_path,
    holdout_metrics_path, forward_calibration_path, holdout_calibration_path,
    tuning_path, feature_contract_path, target_contract_path,
])

celsius_bundle_path, celsius_manifest_path = export_celsius_probability_bundle(
    celsius_bundle,
    celsius_output_dir / "model_weights",
    source_identity={
        "pipeline": "station_training_baseline",
        "notebook": "notebooks/station_training_baseline/stations/Seoul/train_Seoul.ipynb",
        "point_workflow": "Seoul V20 Asia no-peak aligned",
        "probability_setup": "Seoul 1C Market Ordinal Probability Model",
    },
    artifact_paths=celsius_artifact_paths,
)
celsius_manifest = json.loads(celsius_manifest_path.read_text(encoding="utf-8"))
assert celsius_manifest["point_bundle_sha256"] == celsius_sha256_file(exported_weights.bundle_path)
assert celsius_manifest["artifact_integrity"]["bundle_sha256"] == celsius_sha256_file(celsius_bundle_path)
for path in celsius_artifact_paths:
    assert celsius_manifest["artifact_integrity"]["artifact_sha256"][path.name] == celsius_sha256_file(path)
{
    "point_bundle": exported_weights.bundle_path,
    "point_manifest": exported_weights.manifest_path,
    "celsius_probability_bundle": celsius_bundle_path,
    "celsius_probability_manifest": celsius_manifest_path,
    "output_dir": celsius_output_dir,
}
